# Initial setup

## Set API key for Groq
Click [here](https://console.groq.com/keys) to create API key for Groq, if not already created.

In [4]:
import os, json, re, getpass
from dotenv import load_dotenv

load_dotenv( override=True)

False

In [5]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

In [6]:
# if "TEST_API_KEY" not in os.environ:
#     os.environ["TEST_API_KEY"] = getpass.getpass("TEST API Key: ")

In [7]:
from langchain.chat_models import init_chat_model
model_name = "openai/gpt-oss-120b" ##set GPT OSS 120B as the LLM for this lab

# Structured Output Generation Methods

## 0. Without a Method

In [8]:
#Initialize LLM
llm = init_chat_model(model_name, 
                      model_provider="groq")

In [9]:
prompt = """Who won the Champions league in 2022?
            Output should be in JSON and have following fields:
            win_team, lose_team, venue, date, score. Only return the JSON object, nothing else.
         """

In [10]:
llm_response = llm.invoke(prompt)
print(llm_response.content)

{
  "win_team": "Real Madrid",
  "lose_team": "Liverpool",
  "venue": "Stade de France, Saint-Denis, France",
  "date": "2022-05-28",
  "score": "1-0"
}


## 1. Native LLM Output Response Support

In [11]:
#Initialize LLM
llm = init_chat_model(model_name, 
                      model_provider="groq",
                      model_kwargs={"response_format": {"type": "json_object"}})

In [12]:
llm_response = llm.invoke(prompt)
print(llm_response.content)

{"win_team":"Real Madrid","lose_team":"Liverpool","venue":"Stade de France, Saint-Denis, France","date":"2022-05-28","score":"1-0"}


In [13]:
# What would this be?
type(llm_response.content)

str

## 2. Output Parsers

In [14]:
#Initialize LLM without native support
llm = init_chat_model(model_name, 
                      model_provider="groq")

In [15]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

*JsonOutputParser* is a runnable object.

In [16]:
#Example on a sample string
sample_json_str = '{"clarity": "unclear"}'
JsonOutputParser().invoke(sample_json_str)

{'clarity': 'unclear'}

In [17]:
# print(sample_json_str)

In [18]:
#Create a chain
chain = llm | parser

In [19]:
#Get response
llm_response = chain.invoke(prompt)
print(llm_response)

{'win_team': 'Real Madrid', 'lose_team': 'Liverpool', 'venue': 'Stade de France, Saint-Denis, France', 'date': '2022-05-28', 'score': '1-0'}


In [20]:
type(llm_response)

dict

In [21]:
llm_response

{'win_team': 'Real Madrid',
 'lose_team': 'Liverpool',
 'venue': 'Stade de France, Saint-Denis, France',
 'date': '2022-05-28',
 'score': '1-0'}

## 3. Output Parsers With Pydantic

In [22]:
#Initialize LLM without native support
llm = init_chat_model(model_name, 
                      model_provider="groq")

In [23]:
from pydantic import BaseModel, Field

class GameDetails(BaseModel):
    win_team: str = Field(description="The winning team in the football game, and most popular player")
    lose_team: str = Field(description="The losing team in the football game, and most popular player")
    venue: str = Field(description="The venue of the football game, and format should be stadium/venue, city, country")
    date: str = Field(description="The date of the football game, and format should be MMM-YY strictly")
    score: str = Field(description="The score of the football game, and format should losing team score - winning team score; eg. 1-2")



In [24]:

parser = JsonOutputParser(pydantic_object=GameDetails)

In [25]:
#Best practice to add this in the prompt directly
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"win_team": {"description": "The winning team in the football game, and most popular player", "title": "Win Team", "type": "string"}, "lose_team": {"description": "The losing team in the football game, and most popular player", "title": "Lose Team", "type": "string"}, "venue": {"description": "The venue of the football game, and format should be stadium/venue, city, country", "title": "Venue", "type": "string"}, "date": {"description": "The date of the football game, and format should be MMM-YY strictly", "title": "Date", "type

In [26]:
print(prompt)

Who won the Champions league in 2022?
            Output should be in JSON and have following fields:
            win_team, lose_team, venue, date, score. Only return the JSON object, nothing else.
         


In [27]:
print(prompt)

Who won the Champions league in 2022?
            Output should be in JSON and have following fields:
            win_team, lose_team, venue, date, score. Only return the JSON object, nothing else.
         


In [28]:
#Create a chain
chain = llm | parser

In [29]:
#Get response
llm_response = chain.invoke(prompt)
print(llm_response)

{'win_team': 'Real Madrid', 'lose_team': 'Liverpool', 'venue': 'Stade de France, Saint-Denis, France', 'date': '2022-05-28', 'score': '1-0'}


In [30]:
type(llm_response)

dict

In [31]:
llm_response

{'win_team': 'Real Madrid',
 'lose_team': 'Liverpool',
 'venue': 'Stade de France, Saint-Denis, France',
 'date': '2022-05-28',
 'score': '1-0'}

In [32]:
new_prompt = prompt + "\n\n" + "Please return the output in the following JSON format: " + parser.get_format_instructions()

In [33]:
#Get response
llm_response = chain.invoke(new_prompt)
llm_response

{'win_team': 'Real Madrid (Karim Benzema)',
 'lose_team': 'Liverpool (Mohamed Salah)',
 'venue': 'Stade de France, Saint-Denis, France',
 'date': 'May-22',
 'score': '0-1'}

## 4. Structured Output (without parsers)

### With Pydantic

In [34]:
#Initialize LLM without native support
llm = init_chat_model(model_name, 
                      model_provider="groq",
                      temperature=0.0)

In [35]:
prompt = "Who won the Champions league in 2022?"

In [36]:
from pydantic import BaseModel, Field

class GameDetails(BaseModel):
    "Given a user question about a sports event, list the winning team, losing team, venue, date and final score of the game."
    win_team: str = Field(description="The winning team in the football game, and most popular player")
    lose_team: str = Field(description="The losing team in the football game, and most popular player")
    venue: str = Field(description="The venue of the football game, and format should be stadium/venue, city, country")
    date: str = Field(description="The date of the football game, and format should be MMM-YY strictly")
    score: dict = Field(description="The score of the football game, and format should {losing team: score, winning team: score}")

In [37]:
structured_llm = llm.with_structured_output(GameDetails)

In [38]:
response = structured_llm.invoke(prompt)

In [39]:
response

GameDetails(win_team='Real Madrid', lose_team='Liverpool', venue='Stade de France, Saint-Denis, France', date='May-22', score={'Liverpool': 0, 'Real Madrid': 1})

In [40]:
type(response)

__main__.GameDetails

In [41]:
#.model_dump() method converts a model to a dictionary
llm_response = structured_llm.invoke(prompt).model_dump()
llm_response

{'win_team': 'Real Madrid',
 'lose_team': 'Liverpool',
 'venue': 'Stade de France, Saint-Denis, France',
 'date': 'May-22',
 'score': {'Liverpool': 0, 'Real Madrid': 1}}

In [42]:
type(llm_response)

dict

### With TypedDict

In [43]:
#Initialize LLM without native support
llm = init_chat_model(model_name, 
                      model_provider="groq",
                      temperature=0.0)

In [44]:
prompt = "Who won the Champions league in 2022?"

In [45]:
from typing_extensions import Annotated, TypedDict
from typing import Optional

class GameDetails(TypedDict):
    "Given a user question about a sports event, list the winning team, losing team, venue, date and final score of the game."
    win_team: str = Field(description="The winning team in the football game, and most popular player")
    lose_team: str = Field(description="The losing team in the football game, and most popular player")
    venue: str = Field(description="The venue of the football game, and format should be stadium/venue, city, country")
    date: str = Field(description="The date of the football game, and format should be MMM-YY strictly")
    score: dict = Field(description="The score of the football game, and format should {losing team: score, winning team: score}")

In [46]:
structured_llm = llm.with_structured_output(GameDetails)

In [48]:
llm_response = structured_llm.invoke(prompt)
llm_response

BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'Real\u202fMadrid won the 2021‑2022 UEFA Champions League. They defeated Liverpool 1‑0 in the final, which was played on\u202f28\u202fMay\u202f2022 at the Stade\u202fde\u202fFrance in Saint‑Denis, France.'}}

In [ ]:
type(llm_response)

dict

### With JSON Schema

In [ ]:
#Initialize LLM without native support
llm = init_chat_model(model_name, 
                      model_provider="groq",
                      temperature=0.0)

In [ ]:
prompt = "Who won the Champions league in 2022?"

In [ ]:
print(parser.get_format_instructions().split("```")[1])


{"properties": {"win_team": {"description": "The winning team in the football game, and most popular player", "title": "Win Team", "type": "string"}, "lose_team": {"description": "The losing team in the football game, and most popular player", "title": "Lose Team", "type": "string"}, "venue": {"description": "The venue of the football game, and format should be stadium/venue, city, country", "title": "Venue", "type": "string"}, "date": {"description": "The date of the football game, and format should be MMM-YY strictly", "title": "Date", "type": "string"}, "score": {"description": "The score of the football game, and format should losing team score - winning team score; eg. 1-2", "title": "Score", "type": "string"}}, "required": ["win_team", "lose_team", "venue", "date", "score"]}



In [ ]:
json_schema = {
    "title": "GameDetails",
    "description": "Given a user question about a sports event, list the winning team, losing team, venue, date and final score of the game.",
    "type": "object",
    "properties": {
        "win_team": {
            "type": "string",
            "description": "The winning team in the football game, and most popular player"
        },
        "lose_team": {
            "type": "string",
            "description": "The losing team in the football game, and most popular player"
        },
        "venue": {
            "type": "string",
            "description": "The venue of the football game, and format should be stadium/venue, city, country"
        },
        "date": {
            "type": "string",
            "description": "The date of the football game, and format should be MMM-YY strictly"
        },
        "score": {
            "type": "object",
            "description": "The score of the football game, and format should {losing team: score, winning team: score}"
        }
    },
    "required": ["win_team", "lose_team", "venue","date","score"],
}

In [ ]:
structured_llm = llm.with_structured_output(json_schema)

In [ ]:
llm_response = structured_llm.invoke(prompt)
llm_response

{'date': 'May-22',
 'lose_team': 'Liverpool (Mohamed Salah)',
 'score': {'Liverpool': 0, 'Real Madrid': 1},
 'venue': 'Stade de France, Saint-Denis, France',
 'win_team': 'Real Madrid (Karim Benzema)'}

In [ ]:
type(llm_response)

dict